# 🎓 CourseAdvisor AI — Purple Merit Assessment 1
### Run cells one by one in order

In [ ]:
# ── CELL 1: Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

In [ ]:
# ── CELL 2: Install packages ──
!pip install -q streamlit pyngrok langchain langchain-community langchain-groq faiss-cpu sentence-transformers pypdf
print('✅ All packages installed')

In [ ]:
%%writefile /content/app.py
import streamlit as st
import os
import re
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq

# ─────────────────────────────────────────
# Page config
# ─────────────────────────────────────────
st.set_page_config(
    page_title='CourseAdvisor AI',
    page_icon='🎓',
    layout='wide',
    initial_sidebar_state='expanded'
)

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=DM+Serif+Display&family=DM+Sans:wght@300;400;500;600&display=swap');
html, body, [class*='css'] { font-family: 'DM Sans', sans-serif; }
.main-header {
    background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 60%, #0f172a 100%);
    border-radius: 16px; padding: 2rem 2.5rem; margin-bottom: 1.5rem;
    border: 1px solid #1e40af33;
}
.main-header h1 { font-family: 'DM Serif Display', serif; font-size: 2.2rem; color: #f0f9ff; margin: 0 0 0.3rem 0; }
.main-header p  { color: #93c5fd; margin: 0; font-size: 0.95rem; }
.agent-row { display: flex; gap: 0.6rem; flex-wrap: wrap; margin-bottom: 1.5rem; }
.agent-badge { background: #1e293b; border: 1px solid #334155; border-radius: 8px; padding: 0.45rem 0.9rem; font-size: 0.78rem; color: #94a3b8; display: inline-flex; align-items: center; gap: 0.4rem; }
.agent-badge.active { background: #1e3a5f; border-color: #3b82f6; color: #93c5fd; }
.agent-badge .dot { width: 7px; height: 7px; border-radius: 50%; background: #64748b; }
.agent-badge.active .dot { background: #3b82f6; box-shadow: 0 0 6px #3b82f6; animation: pulse 1.5s infinite; }
@keyframes pulse { 0%,100%{opacity:1} 50%{opacity:0.4} }
.section-card { background: #0f172a; border: 1px solid #1e293b; border-radius: 12px; padding: 1.2rem 1.5rem; margin-bottom: 1rem; }
.section-card h3 { color: #e2e8f0; font-size: 0.88rem; font-weight: 600; letter-spacing: 0.08em; text-transform: uppercase; margin: 0 0 0.8rem 0; }
.answer-block { background: #0f172a; border-left: 3px solid #3b82f6; border-radius: 0 12px 12px 0; padding: 1.4rem 1.6rem; margin-top: 1rem; color: #cbd5e1; font-size: 0.93rem; line-height: 1.75; white-space: pre-wrap; }
.verify-ok   { background: #052e16; border: 1px solid #16a34a; border-radius: 10px; padding: 0.9rem 1.2rem; color: #86efac; font-size: 0.85rem; margin-top: 0.8rem; }
.verify-warn { background: #431407; border: 1px solid #ea580c; border-radius: 10px; padding: 0.9rem 1.2rem; color: #fdba74; font-size: 0.85rem; margin-top: 0.8rem; }
.step-row { display: flex; align-items: center; gap: 0.6rem; padding: 0.5rem 0; color: #64748b; font-size: 0.85rem; border-bottom: 1px solid #1e293b; }
.step-row:last-child { border-bottom: none; }
.step-row.done    { color: #4ade80; }
.step-row.running { color: #93c5fd; }
.step-icon { width: 18px; text-align: center; }
.info-box { background: #1e3a5f22; border: 1px solid #3b82f640; border-radius: 10px; padding: 1rem 1.2rem; color: #93c5fd; font-size: 0.88rem; margin-bottom: 1rem; }
[data-testid='stSidebar'] { background: #0f172a; border-right: 1px solid #1e293b; }
[data-testid='stSidebar'] label, [data-testid='stSidebar'] p { color: #94a3b8 !important; font-size: 0.85rem !important; }
[data-testid='stSidebar'] h2, [data-testid='stSidebar'] h3 { color: #e2e8f0 !important; }
.stTextInput>div>div>input, .stTextArea>div>div>textarea { background: #1e293b !important; border: 1px solid #334155 !important; border-radius: 8px !important; color: #e2e8f0 !important; }
.stButton>button { background: linear-gradient(135deg, #1d4ed8, #2563eb) !important; color: white !important; border: none !important; border-radius: 8px !important; padding: 0.55rem 1.5rem !important; font-weight: 500 !important; }
</style>
""", unsafe_allow_html=True)


# ─────────────────────────────────────────
# Ingest helpers (from ingest.py)
# ─────────────────────────────────────────
def get_university(filename):
    name = filename.lower()
    if 'pucit'    in name: return 'PUCIT'
    if 'umt'      in name: return 'UMT'
    if 'fast'     in name: return 'FAST'
    if 'nust'     in name: return 'NUST'
    if 'sargodah' in name: return 'Sargodah'
    if 'malakand' in name: return 'Malakand'
    if 'iit'      in name: return 'IIT'
    if 'usa'      in name: return 'USA'
    return 'Unknown'

def get_program(filename):
    name = filename.lower()
    if 'data-science' in name or 'data science' in name or 'ds' in name: return 'DS'
    if 'software-engineering' in name or 'se' in name or 'bese' in name or 'bsse' in name: return 'SE'
    if 'information-technology' in name or 'it' in name: return 'IT'
    if 'cs' in name or 'cse' in name: return 'CS'
    if 'rule' in name or 'regulation' in name or 'handbook' in name: return 'POLICY'
    return 'GENERAL'

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\.{3,}', ' ', text)
    text = re.sub(r'\x0c', ' ', text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    return text.strip()

def extract_prereq_text(text):
    lines = text.split('\n')
    enriched = []
    pattern = re.compile(
        r'([A-Z]{2,4}[-\s]?\d{3,4})\s+([A-Za-z\s&/]{5,50}?)\s+([A-Z]{2,4}[-\s]?\d{3,4})',
        re.IGNORECASE
    )
    for line in lines:
        match = pattern.search(line)
        if match:
            code   = match.group(1).strip()
            title  = match.group(2).strip()
            prereq = match.group(3).strip()
            enriched.append(
                f'Course {code} ({title}) requires {prereq} as a prerequisite. '
                f'To enroll in {title} you must have completed {prereq}.'
            )
        else:
            enriched.append(line)
    return '\n'.join(enriched)


# ─────────────────────────────────────────
# Index builder
# ─────────────────────────────────────────
@st.cache_resource(show_spinner=False)
def get_embeddings():
    return HuggingFaceEmbeddings(
        model_name='sentence-transformers/all-MiniLM-L6-v2',
        model_kwargs={'device': 'cpu'}
    )

def build_index_from_drive(data_path, index_path):
    embeddings = get_embeddings()
    all_docs, log = [], []
    pdf_files = [f for f in os.listdir(data_path) if f.endswith('.pdf')]
    if not pdf_files:
        return None, ['❌ No PDF files found in the data folder.'], 0
    for filename in pdf_files:
        path = os.path.join(data_path, filename)
        try:
            loader = PyPDFLoader(path)
            pages  = loader.load()
            univ   = get_university(filename)
            prog   = get_program(filename)
            for page in pages:
                page.page_content = clean_text(page.page_content)
                page.page_content = extract_prereq_text(page.page_content)
                page.metadata['source']     = filename
                page.metadata['university'] = univ
                page.metadata['program']    = prog
            all_docs.extend(pages)
            log.append(f'✅ {filename} ({len(pages)} pages) [{univ} | {prog}]')
        except Exception as e:
            log.append(f'❌ {filename}: {e}')
    if not all_docs:
        return None, log + ['❌ No documents loaded successfully.'], 0
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800, chunk_overlap=200,
        separators=['\n\n', '\n', '. ', ' ', '']
    )
    chunks = splitter.split_documents(all_docs)
    vs = FAISS.from_documents(chunks, embeddings)
    os.makedirs(index_path, exist_ok=True)
    vs.save_local(index_path)
    return vs, log, len(chunks)

def load_existing_index(index_path):
    return FAISS.load_local(
        index_path, get_embeddings(),
        allow_dangerous_deserialization=True
    )


# ─────────────────────────────────────────
# AGENT 1: Intake (from assistant.py)
# ─────────────────────────────────────────
def intake_agent(query, completed, program, university, term, credits):
    query_lower = query.lower()
    is_prereq_query = any(w in query_lower for w in [
        'can i take', 'eligible', 'prerequisite', 'prereq',
        'completed', 'enroll', 'register', 'grade'
    ])
    is_general_query = any(w in query_lower for w in [
        'how many credits', 'total credits', 'core courses',
        'program require', 'capstone', 'elective', 'fee',
        'tuition', 'professor', 'available', 'online',
        'class size', 'minimum cgpa', 'policy', 'waiver',
        'trace', 'path', 'chain', 'fastest', 'need before'
    ])
    if is_general_query:
        if not university or not program:
            return 'MISSING: Please tell me your university and program (CS/IT/SE/DS).'
        return 'PROFILE_COMPLETE'
    if is_prereq_query:
        missing = []
        if not university: missing.append('Which university are you from?')
        if not program:    missing.append('What is your program? (CS/IT/SE/DS)')
        if not completed:  missing.append('Which courses have you already completed?')
        if missing:
            return 'MISSING: ' + ' | '.join(missing)
        return 'PROFILE_COMPLETE'
    if not university or not program:
        return 'MISSING: Please provide your university and program.'
    return 'PROFILE_COMPLETE'


# ─────────────────────────────────────────
# AGENT 2: Retriever (from assistant.py)
# ── FIX: hard block cross-university fallback ──
# ─────────────────────────────────────────
def retriever_agent(vectorstore, query, university, k=7):
    query_lower = query.lower()
    if any(w in query_lower for w in ['can i take', 'eligible', 'enroll', 'register']):
        search_query = f'{university} requires prerequisite {query}'
    elif any(w in query_lower for w in ['chain', 'path', 'before', 'need', 'fastest']):
        search_query = f'{university} prerequisite chain {query}'
    elif any(w in query_lower for w in ['policy', 'rule', 'regulation', 'cgpa', 'credit', 'fee', 'waiver']):
        search_query = f'{university} policy rule regulation {query}'
    else:
        search_query = f'{university} {query}'

    retriever = vectorstore.as_retriever(
        search_type='similarity',
        search_kwargs={'k': k}
    )
    docs = retriever.invoke(search_query)

    # ── HARD BLOCK: only use docs from the requested university ──
    university_docs = [
        d for d in docs
        if university.upper() in d.metadata.get('university', '').upper()
    ]

    # ── FIX: if no university-specific docs found, return abstain chunk ──
    # Never fall back to other university documents
    if len(university_docs) == 0:
        return [{
            'id': 1,
            'text': (
                f'No catalog information found for {university}. '
                f'This information is not available in the provided {university} catalog documents. '
                f'Please check the official {university} academic policy or contact your academic advisor.'
            ),
            'source': 'NO_SOURCE',
            'university': university,
            'program': 'N/A',
            'page': 'N/A',
        }]

    # Use university-specific docs if >= 2, otherwise use all matches found
    final_docs = university_docs if len(university_docs) >= 2 else university_docs

    return [{
        'id': i + 1,
        'text': doc.page_content,
        'source': doc.metadata.get('source', 'unknown'),
        'university': doc.metadata.get('university', 'unknown'),
        'program': doc.metadata.get('program', 'unknown'),
        'page': doc.metadata.get('page', '?'),
    } for i, doc in enumerate(final_docs)]


# ─────────────────────────────────────────
# Format chunks for prompt
# ─────────────────────────────────────────
def format_chunks(chunks):
    out = ''
    for c in chunks:
        out += f"\n[CHUNK {c['id']}] {c['source']} | Page {c['page']} | {c['university']} | {c['program']}\n"
        out += c['text'] + '\n'
        out += '-' * 50
    return out


# ─────────────────────────────────────────
# AGENT 3: Planner (from assistant.py)
# ─────────────────────────────────────────
def planner_agent(llm, query, completed, program, university, term, credits, chunks):
    prompt = f"""
You are a strict academic advisor AI.
You ONLY use the catalog excerpts below. NEVER use outside knowledge.
If information is not in the excerpts, say exactly:
"I don't have that information in the provided catalog/policies."
Then suggest: Check the official university website or contact your academic advisor.

STUDENT PROFILE:
- University: {university}
- Program: {program}
- Completed Courses: {completed or 'None'}
- Target Term: {term}
- Max Credits: {credits}
- Query: {query}

CATALOG EXCERPTS:
{format_chunks(chunks)}

RULES:
1. Cite EVERY claim like this: [Source: filename, Page X]
2. For prerequisite questions state: ELIGIBLE / NOT ELIGIBLE / NEED MORE INFO
3. For eligibility check if completed courses satisfy the prereq listed in excerpts
4. If student grade is D or below state NOT ELIGIBLE (minimum C required)
5. Show reasoning step by step
6. NEVER use information from a different university's documents
7. If source is NO_SOURCE, you MUST say: "I don't have that information in the provided catalog/policies."

REQUIRED OUTPUT FORMAT — use exactly these 5 sections:

Answer / Plan:
[your answer]

Why (requirements/prereqs satisfied):
[reasoning with citations]

Citations:
[Source: file, Page X — what it says]

Clarifying questions (if needed):
[questions or write: None]

Assumptions / Not in catalog:
[what you assumed or could not find]
"""
    return llm.invoke(prompt).content


# ─────────────────────────────────────────
# AGENT 4: Verifier (from assistant.py)
# ─────────────────────────────────────────
def verifier_agent(llm, response):
    prompt = f"""
Check this academic advisor response:

{response}

Verify:
1. Does every factual claim have a [Source: ...] citation?
2. Are all 5 required sections present?
3. Any hallucinated facts not from the excerpts?
4. Does the response use information from the correct university only?

Output either:
VERIFIED: Response is properly cited and complete.
OR
ISSUES: [list problems found]
"""
    return llm.invoke(prompt).content


# ─────────────────────────────────────────
# Session state
# ─────────────────────────────────────────
for k, v in {
    'vectorstore': None,
    'index_built': False,
    'chat_history': [],
    'total_chunks': 0,
    'ingest_log': [],
}.items():
    if k not in st.session_state:
        st.session_state[k] = v


# ─────────────────────────────────────────
# SIDEBAR
# ─────────────────────────────────────────
with st.sidebar:
    st.markdown('## ⚙️ Configuration')
    st.markdown('---')
    groq_key = st.text_input('🔑 Groq API Key', type='password', placeholder='gsk_...')
    st.markdown('---')
    st.markdown('### 📁 Google Drive Paths')
    data_path  = st.text_input('Data folder (PDFs)',  value='/content/drive/MyDrive/rag_project/data')
    index_path = st.text_input('FAISS index folder',  value='/content/drive/MyDrive/rag_project/faiss_index')
    st.markdown('---')
    col1, col2 = st.columns(2)
    with col1:
        if st.button('🚀 Build Index', use_container_width=True):
            if not os.path.exists(data_path):
                st.error('Path not found. Make sure Drive is mounted.')
            else:
                with st.spinner('Ingesting PDFs from Drive...'):
                    vs, log, n = build_index_from_drive(data_path, index_path)
                if vs:
                    st.session_state.vectorstore  = vs
                    st.session_state.index_built  = True
                    st.session_state.total_chunks = n
                    st.session_state.ingest_log   = log
                    st.success(f'✅ {n} chunks indexed!')
                else:
                    st.error('Build failed. Check log.')
                    st.session_state.ingest_log = log
    with col2:
        if st.button('📂 Load Index', use_container_width=True):
            if not os.path.exists(index_path):
                st.error('Index not found.')
            else:
                with st.spinner('Loading index...'):
                    try:
                        vs = load_existing_index(index_path)
                        st.session_state.vectorstore = vs
                        st.session_state.index_built = True
                        st.success('✅ Index loaded!')
                    except Exception as e:
                        st.error(f'Failed: {e}')
    if st.session_state.ingest_log:
        with st.expander('📋 Ingest Log'):
            for line in st.session_state.ingest_log:
                st.text(line)
    st.markdown('---')
    st.markdown('### 🎓 Student Profile')
    university = st.text_input('University', placeholder='e.g. FAST, NUST, PUCIT')
    program    = st.selectbox('Program', ['CS', 'SE', 'IT', 'DS', 'GENERAL'])
    completed  = st.text_area('Completed Courses', placeholder='e.g. CS101, MATH101, CS201', height=80)
    term       = st.selectbox('Target Term', ['Fall', 'Spring', 'Summer'])
    credits    = st.selectbox('Max Credits', ['18', '15', '12', '21'])
    st.markdown('---')
    if st.button('🗑️ Clear Chat', use_container_width=True):
        st.session_state.chat_history = []
        st.rerun()
    if st.session_state.index_built:
        st.markdown('---')
        st.markdown(f"**📊 Index:** `{st.session_state.total_chunks}` chunks" if st.session_state.total_chunks else '**📊 Index loaded**')


# ─────────────────────────────────────────
# MAIN AREA
# ─────────────────────────────────────────
st.markdown('<div class="main-header"><h1>🎓 CourseAdvisor AI</h1><p>Agentic RAG · Catalog-Grounded · 4-Agent Pipeline · Purple Merit Assessment 1</p></div>', unsafe_allow_html=True)

idx_built = st.session_state.index_built
active = 'active' if idx_built else ''
st.markdown(f"""
<div class="agent-row">
  <span class="agent-badge {active}"><span class="dot"></span> Agent 1 · Intake</span>
  <span class="agent-badge {active}"><span class="dot"></span> Agent 2 · Retriever</span>
  <span class="agent-badge {active}"><span class="dot"></span> Agent 3 · Planner</span>
  <span class="agent-badge {active}"><span class="dot"></span> Agent 4 · Verifier</span>
</div>
""", unsafe_allow_html=True)

if not groq_key:
    st.markdown('<div class="info-box">👈 Enter your <b>Groq API key</b> in the sidebar to get started.</div>', unsafe_allow_html=True)
    st.stop()

if not st.session_state.index_built:
    st.markdown('<div class="info-box">👈 Click <b>Build Index</b> (first time) or <b>Load Index</b> (already built) in the sidebar.</div>', unsafe_allow_html=True)
    st.stop()

# Chat history display
for entry in st.session_state.chat_history:
    with st.chat_message('user'):
        st.markdown(entry['query'])
    with st.chat_message('assistant'):
        st.markdown(f'<div class="answer-block">{entry["answer"]}</div>', unsafe_allow_html=True)
        if entry.get('verification'):
            cls = 'verify-ok' if 'VERIFIED' in entry['verification'] else 'verify-warn'
            st.markdown(f'<div class="{cls}">🔍 <b>Verifier:</b> {entry["verification"]}</div>', unsafe_allow_html=True)

# Chat input
query = st.chat_input('Ask about prerequisites, course plans, policies...')

if query:
    with st.chat_message('user'):
        st.markdown(query)

    with st.chat_message('assistant'):
        steps_ph = st.empty()

        def show_steps(step_idx):
            labels = [
                'Agent 1 · Intake — checking profile',
                'Agent 2 · Retriever — searching catalog',
                'Agent 3 · Planner — generating answer',
                'Agent 4 · Verifier — checking citations',
            ]
            html = ''
            for i, label in enumerate(labels):
                done    = i < step_idx
                running = i == step_idx
                cls     = 'done' if done else ('running' if running else '')
                icon    = '✅' if done else ('⟳' if running else '○')
                html   += f'<div class="step-row {cls}"><span class="step-icon">{icon}</span>{label}</div>'
            steps_ph.markdown(f'<div class="section-card"><h3>Pipeline</h3>{html}</div>', unsafe_allow_html=True)

        # Agent 1
        show_steps(0)
        intake = intake_agent(query, completed, program, university, term, credits)

        if 'MISSING' in intake:
            steps_ph.empty()
            msg = intake.replace('MISSING: ', '')
            st.warning(f'⚠️ Profile incomplete. Please fill in the sidebar:\n\n{msg}')
            st.session_state.chat_history.append({'query': query, 'answer': msg, 'verification': None})
            st.stop()

        # Agent 2
        show_steps(1)
        chunks = retriever_agent(st.session_state.vectorstore, query, university)

        # Agent 3
        show_steps(2)
        llm    = ChatGroq(api_key=groq_key, model_name='llama-3.1-8b-instant', temperature=0)
        answer = planner_agent(llm, query, completed, program, university, term, credits, chunks)

        # Agent 4
        show_steps(3)
        verification = verifier_agent(llm, answer)

        steps_ph.empty()

        st.markdown(f'<div class="answer-block">{answer}</div>', unsafe_allow_html=True)
        cls = 'verify-ok' if 'VERIFIED' in verification else 'verify-warn'
        st.markdown(f'<div class="{cls}">🔍 <b>Verifier:</b> {verification}</div>', unsafe_allow_html=True)

        with st.expander(f'📄 Retrieved {len(chunks)} catalog chunks'):
            for c in chunks:
                st.markdown(f"**[Chunk {c['id']}]** `{c['source']}` · Page {c['page']} · {c['university']} · {c['program']}")
                st.caption(c['text'][:300] + '...')

        st.session_state.chat_history.append({
            'query': query,
            'answer': answer,
            'verification': verification,
        })

In [ ]:
# ── CELL 4: Launch Streamlit + ngrok ──
import subprocess, time, requests
from pyngrok import ngrok

# ── PASTE YOUR NGROK TOKEN HERE ──
# Free token at: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = 'PASTE_YOUR_NGROK_TOKEN_HERE'

!ngrok authtoken {NGROK_TOKEN}

# Kill any old tunnels
ngrok.kill()

# Start Streamlit as background process
proc = subprocess.Popen(
    [
        'streamlit', 'run', '/content/app.py',
        '--server.port=8501',
        '--server.headless=true',
        '--server.enableCORS=false',
        '--server.enableXsrfProtection=false',
        '--server.fileWatcherType=none',
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Wait until Streamlit is actually up
print('⏳ Waiting for Streamlit to start...')
for i in range(20):
    time.sleep(2)
    try:
        r = requests.get('http://localhost:8501', timeout=2)
        if r.status_code == 200:
            print('✅ Streamlit is running!')
            break
    except:
        print(f'   still starting... {(i+1)*2}s')

# Open ngrok tunnel
tunnel = ngrok.connect(8501)
print()
print('=' * 55)
print('🎓  OPEN THIS URL IN YOUR BROWSER:')
print(f'    {tunnel.public_url}')
print('=' * 55)
print('Keep this cell running to keep the app alive.')